In [1]:
!pip install datasets timm torch torchvision pillow numpy

In [2]:
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import json

# --- 1. DEVICE & DATASET LOADING ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Downloading dataset from Hugging Face...")
# Loading the dataset without specifying a split downloads the full dictionary of splits
hf_dataset = load_dataset("c3rl/IIIT-INDIC-HW-WORDS-Hindi")

# Access the predefined official splits directly
train_data = hf_dataset['train']
val_data = hf_dataset['validation']
test_data = hf_dataset['test']

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Testing samples: {len(test_data)}")

# --- 2. DYNAMIC VOCABULARY BUILDER ---
print("Building dynamic vocabulary from the training dataset...")
unique_chars = set()
for item in train_data:
    text = item['text']
    if text:
        unique_chars.update(list(text))

# CTC Loss REQUIRES the 0th index to be a <BLANK> token for sequence alignment
vocab = ["<BLANK>"] + sorted(list(unique_chars))
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}
num_classes = len(vocab)
print(f"Total Unique Characters (including blank): {num_classes}")

# Save the vocabulary mapping immediately so the FastAPI backend can use it later
with open('ctc_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(idx_to_char, f, ensure_ascii=False, indent=4)
print("Saved vocabulary to ctc_vocab.json")

# --- 3. CUSTOM PYTORCH DATASET ---
class HFWordDataset(Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = item['image'].convert('RGB')
        text = item['text']

        if self.transform:
            image = self.transform(image)

        # Convert the string of text into a sequence of integer indices
        target = [char_to_idx[c] for c in text if c in char_to_idx]
        target_length = len(target)

        return image, target, target_length

# Standardize word images to Height=64, Width=384
transform = transforms.Compose([
    transforms.Resize((64, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = HFWordDataset(train_data, transform=transform)
val_dataset = HFWordDataset(val_data, transform=transform)
test_dataset = HFWordDataset(test_data, transform=transform)

# --- 4. THE CTC COLLATE FUNCTION ---
def ctc_collate_fn(batch):
    """
    PyTorch CTC Loss expects targets to be a flat 1D tensor,
    and requires a separate tensor containing the lengths of each target.
    """
    images, targets, target_lengths = [], [], []
    for img, tgt, tgt_len in batch:
        images.append(img)
        targets.extend(tgt) # extend() flattens the lists into one long 1D array
        target_lengths.append(tgt_len)

    images = torch.stack(images)
    targets = torch.tensor(targets, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)

    return images, targets, target_lengths

# Batch size of 32 is standard; adjust to 16 if your GPU runs out of memory
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=ctc_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=ctc_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=ctc_collate_fn)

# --- 5. THE HYBRID SEQUENCE ARCHITECTURE ---
class HybridCTCModel(nn.Module):
    def __init__(self, num_classes):
        super(HybridCTCModel, self).__init__()

        resnet = timm.create_model('resnet18', pretrained=True)
        # Use up to layer 3. Layer 4 downsamples too much, leaving too few time-steps for long words.
        self.cnn = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.act1, resnet.maxpool, # downsamples by 4
            resnet.layer1, # downsamples by 4
            resnet.layer2, # downsamples by 8
            resnet.layer3  # downsamples by 16
        )

        # Input height is 64. Downsampled by 16 = height 4.
        # Input width is 384. Downsampled by 16 = width 24 (24 time-steps).
        # We squash the height from 4 to 1 to create a 1D sequence of features.
        self.sequence_builder = nn.Conv2d(256, 256, kernel_size=(4, 1))

        # Transformer analyzes the sequence to understand global context
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=256, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=2)

        # Final prediction layer for the vocabulary
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        # x shape: [Batch, Channels, Height, Width] -> [B, 3, 64, 384]
        x = self.cnn(x)                # [B, 256, 4, 24]
        x = self.sequence_builder(x)   # [B, 256, 1, 24]

        # Squeeze height and swap axes for Transformer: [Batch, Time_Steps, Features]
        x = x.squeeze(2).permute(0, 2, 1) # [B, 24, 256]

        # Transformer Context Mapping
        x = self.transformer(x)

        # Vocabulary mapping
        x = self.fc(x)                 # [B, 24, Num_Classes]

        # PyTorch CTCLoss requires shape: [Time_Steps, Batch, Num_Classes]
        return x.log_softmax(2).permute(1, 0, 2)

# --- 6. INITIALIZE TRAINING ---
model = HybridCTCModel(num_classes).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True) # zero_infinity prevents crashes on extreme target lengths
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

# --- 7. TRAINING LOOP ---
epochs = 10
seq_length = 24 # Derived from input width (384) divided by CNN downsampling factor (16)

print("Starting End-to-End Sequence Training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, (images, targets, target_lengths) in enumerate(train_loader):
        images = images.to(device)
        targets = targets.to(device)

        batch_size = images.size(0)

        # Create a tensor indicating the length of the predicted sequence for every image in the batch
        input_lengths = torch.full(size=(batch_size,), fill_value=seq_length, dtype=torch.long).to(device)

        optimizer.zero_grad()
        preds = model(images) # Shape: [24, Batch, Num_Classes]

        loss = criterion(preds, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print progress every 100 batches to monitor training speed
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, targets, target_lengths in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            batch_size = images.size(0)
            input_lengths = torch.full(size=(batch_size,), fill_value=seq_length, dtype=torch.long).to(device)

            preds = model(images)
            loss = criterion(preds, targets, input_lengths, target_lengths)
            val_loss += loss.item()

    print(f"==== Epoch [{epoch+1}/{epochs}] Summary ====")
    print(f"Train Loss: {running_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")
    print("====================================\n")

# --- 8. SAVE FINAL MODEL ---
torch.save(model.state_dict(), 'hybrid_ctc_model.pth')
print("Training Complete! Saved hybrid_ctc_model.pth")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/459M [00:00<?, ?B/s]

... (5 outputs left out)

Training samples: 69853
Validation samples: 12708
Testing samples: 12869
Building dynamic vocabulary from the training dataset...
Total Unique Characters (including blank): 109
Saved vocabulary to ctc_vocab.json


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Starting End-to-End Sequence Training...
Epoch [1/10], Step [100/2183], Loss: 4.1206
Epoch [1/10], Step [200/2183], Loss: 3.7378
Epoch [1/10], Step [300/2183], Loss: 2.6313
Epoch [1/10], Step [400/2183], Loss: 1.7116
Epoch [1/10], Step [500/2183], Loss: 1.2901
Epoch [1/10], Step [600/2183], Loss: 1.2842
Epoch [1/10], Step [700/2183], Loss: 1.1707
Epoch [1/10], Step [800/2183], Loss: 0.7454
Epoch [1/10], Step [900/2183], Loss: 0.7970
Epoch [1/10], Step [1000/2183], Loss: 0.9752
Epoch [1/10], Step [1100/2183], Loss: 0.7727
Epoch [1/10], Step [1200/2183], Loss: 0.6386
Epoch [1/10], Step [1300/2183], Loss: 0.6822
Epoch [1/10], Step [1400/2183], Loss: 0.5913
Epoch [1/10], Step [1500/2183], Loss: 0.3760
Epoch [1/10], Step [1600/2183], Loss: 0.7647
Epoch [1/10], Step [1700/2183], Loss: 0.4575
Epoch [1/10], Step [1800/2183], Loss: 0.5504
Epoch [1/10], Step [1900/2183], Loss: 0.4429
Epoch [1/10], Step [2000/2183], Loss: 0.4963
Epoch [1/10], Step [2100/2183], Loss: 0.4772
==== Epoch [1/10] Summa

KeyboardInterrupt: 

In [3]:
import torch
import json

# Save the trained model weights that are currently in memory
torch.save(model.state_dict(), 'hybrid_ctc_model.pth')
print("Model safely saved after early stopping!")

# Ensure the vocabulary is also saved
with open('ctc_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(idx_to_char, f, ensure_ascii=False, indent=4)
print("Vocabulary safely saved!")

Model safely saved after early stopping!
Vocabulary safely saved!
